In [42]:
%pip install sounddevice numpy librosa matplotlib scipy openai python-dotenv ipykernel

Note: you may need to restart the kernel to use updated packages.


In [43]:
%pip install openai pydub


Note: you may need to restart the kernel to use updated packages.


In [44]:
# Step 3: Basic Transcription (Without Chunking)
# Imports and API connection

import numpy as np
import librosa
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
import os
from IPython.display import Audio, display
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("✅ OpenAI client ready")

✅ OpenAI client ready


In [45]:
# Find where the notebook runs and where the MP3 files are
import os
import glob

print("Notebook folder:", os.getcwd())
print("Files and folders here:", os.listdir())
print("MP3 files found in this folder and subfolders:", glob.glob("**/*.mp3", recursive=True))

Notebook folder: /Users/macbook-air/Desktop/AI consulting/Week1/Day5/lab-whisper-stt
Files and folders here: ['.DS_Store', 'README.md', 'chunks', 'audio', 'transcriptions', '.gitignore', '.env', 'project_nimbus_meeting.mp3', 'rubric.md', 'whisper_transcription.ipynb', '.git', 'instructions.md', 'CA138clip.mp3']
MP3 files found in this folder and subfolders: ['project_nimbus_meeting.mp3', 'CA138clip.mp3', 'audio/project_nimbus_meeting.mp3']


In [46]:
# Load the first 30 seconds as numbers, the same format as the microphone recordings
audio_path = r"/Users/macbook-air/Downloads/CA138clip.mp3"
sample_rate = 16000
duration = 30  # seconds: transcribe only a short part of the file

print(f"🎧 Loading the first {duration} seconds of {audio_path}...")
audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True, duration=duration)
print("✅ Audio loaded!")

display(Audio(audio, rate=sample_rate))

🎧 Loading the first 30 seconds of /Users/macbook-air/Downloads/CA138clip.mp3...
✅ Audio loaded!


In [47]:
# Step 3 (continued): Basic Transcription (Without Chunking)

audio_int16 = np.int16(audio * 32767)   # Prepare audio

# Create in-memory WAV file
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "short_clip.wav"  # Whisper needs a filename

# 3.1 Transcribe a short part of the audio file
print("🤖 Transcribing with Whisper...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    response_format="verbose_json"
)

# 3.2 Understand the API response format
print("\n🔍 Response type:", type(transcript).__name__)
print("Fields in the response:", list(transcript.model_dump().keys()))

# 3.3 Extract basic transcription text
print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

print(f"\n🌍 Language detected: {transcript.language}")
print(f"⏱️ Duration transcribed: {transcript.duration:.2f}s")

🤖 Transcribing with Whisper...



🔍 Response type: TranscriptionVerbose
Fields in the response: ['duration', 'language', 'text', 'segments', 'usage', 'words', 'task']

📝 Transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now.

🌍 Language detected: english
⏱️ Duration transcribed: 30.00s


In [48]:
# Step 4: Transcription with Prompts (Guided Approach)
# 4.1 Create prompts with context (technical terms, names, etc.)

# Keep the Step 3 result as the "without context" version for the comparison
text_without_context = transcript.text

# Words Whisper should expect: the speakers talk about the Apollo moon landings
context_prompt = (
    "Interview from the Dictionary of American Regional English. "
    "The speakers talk about the Apollo moon landings: astronauts, NASA, "
    "Neil Armstrong, Buzz Aldrin, the moon, lunar module, moon rocks, "
    "kangaroo hop, splashdown, back to Earth."
)

print("📋 Context prompt:")
print("-" * 40)
print(context_prompt)

📋 Context prompt:
----------------------------------------
Interview from the Dictionary of American Regional English. The speakers talk about the Apollo moon landings: astronauts, NASA, Neil Armstrong, Buzz Aldrin, the moon, lunar module, moon rocks, kangaroo hop, splashdown, back to Earth.


In [49]:
# 4.2 Transcribe with prompts

buffer.seek(0)  # Rewind: the buffer was already read by the Step 3 call
print("🤖 Transcribing with context...")
transcript_prompted = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    prompt=context_prompt
)
text_with_context = transcript_prompted.text  # Saved for the comparison below

print("\n📝 Transcription:")
print("-" * 40)
print(text_with_context)

# Save both versions to text files
os.makedirs("transcriptions", exist_ok=True)

with open("transcriptions/step4_without_prompt.txt", "w", encoding="utf-8") as f:
    f.write(text_without_context)

with open("transcriptions/step4_with_prompt.txt", "w", encoding="utf-8") as f:
    f.write(text_with_context)

print("\n💾 Saved to 'transcriptions/step4_without_prompt.txt' and 'transcriptions/step4_with_prompt.txt'")

🤖 Transcribing with context...

📝 Transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around. What did they call it? Kangaroo walk? Something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Oh, yes. I'll be so glad when they land back now. I think that's pretty well a fact, because they've landed so many safely now.

💾 Saved to 'transcriptions/step4_without_prompt.txt' and 'transcriptions/step4_with_prompt.txt'


In [50]:
# 4.3 Compare results

print("=" * 50)
print("WITHOUT PROMPT (Step 3)")
print("=" * 50)
print(text_without_context)

print("\n" + "=" * 50)
print("WITH PROMPT (Step 4)")
print("=" * 50)
print(text_with_context)

WITHOUT PROMPT (Step 3)
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now.

WITH PROMPT (Step 4)
It was rather interesting just to watch them gathering their materials and bouncing around. What did they call it? Kangaroo walk? Something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Oh, yes. I'll be so glad when they land back now. I think that's pretty well a fact, because they've landed so many safely now.


In [51]:
# 4.3 Compare results
import re
import html
import difflib
from IPython.display import HTML, display

def tokenize(text):
    """Split text into words, keeping numbers like 7.2 together."""
    return re.findall(r"\d+(?:\.\d+)?|[\w'’-]+", text)

# Every place where the two transcripts differ
before, after = tokenize(text_without_context), tokenize(text_with_context)
rows = ""
for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, before, after).get_opcodes():
    if op != "equal":
        old = html.escape(" ".join(before[i1:i2])) or "—"
        new = html.escape(" ".join(after[j1:j2])) or "—"
        rows += f"<tr><td style='color:#c0392b'>{old}</td><td style='color:#1e8449'>{new}</td></tr>"

display(HTML(f"""
<h3>Word count</h3>
<p>Without context: <b>{len(before)}</b> words &nbsp;→&nbsp; With context: <b>{len(after)}</b> words</p>
<h3>Where the two transcripts differ</h3>
<table>
  <tr><th>Without context</th><th>With context</th></tr>
  {rows or "<tr><td colspan='2'>No differences</td></tr>"}
</table>
"""))

Without context,With context
their what,What did
kangaroo,Kangaroo
or something,Something
Maine,Earth
But,—


### Step 4.3: Comparing the results

**Without context (Step 3):**
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now.

**With context (Step 4):**
It was rather interesting just to watch them gathering their materials and bouncing around their,  what they call it, kangaroo walk or something like that. 
Who named it that?  I don't know. I bet those men are going to get quite a reception when they get back to Maine.  Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now.

**My observations**
- The prompt about the Apollo moon landings [did / did not] change the words related to that topic.
- The words that changed were amde sentences better, grammatically.
- Without a reference text, I could only judge accuracy by listening, so this comparison is my own judgement, not a measured error rate.

In [52]:
# Step 5: Transcription Without Prompts (Unguided Approach)
# 5.1 Transcribe without any prompt

buffer.seek(0)  # Rewind: the buffer was already read by the earlier calls
print("🤖 Transcribing without context...")
transcript_unguided = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer
)
text_unguided = transcript_unguided.text  # Saved for the comparison below

print("\n📝 Transcription:")
print("-" * 40)
print(text_unguided)

# Save the result
with open("transcriptions/step5_without_prompt.txt", "w", encoding="utf-8") as f:
    f.write(text_unguided)

print("\n💾 Saved to 'transcriptions/step5_without_prompt.txt'")

🤖 Transcribing without context...

📝 Transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now.

💾 Saved to 'transcriptions/step5_without_prompt.txt'


In [53]:
# 5.2 Compare with prompted results

print("=" * 50)
print("WITHOUT PROMPT (unguided)")
print("=" * 50)
print(text_unguided)

print("\n" + "=" * 50)
print("WITH PROMPT (guided)")
print("=" * 50)
print(text_with_context)

WITHOUT PROMPT (unguided)
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now.

WITH PROMPT (guided)
It was rather interesting just to watch them gathering their materials and bouncing around. What did they call it? Kangaroo walk? Something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Oh, yes. I'll be so glad when they land back now. I think that's pretty well a fact, because they've landed so many safely now.


In [54]:
# 5.3 Analyze differences (functions defined in Step 4.3)

before, after = tokenize(text_unguided), tokenize(text_with_context)
rows = ""
for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, before, after).get_opcodes():
    if op != "equal":
        old = html.escape(" ".join(before[i1:i2])) or "—"
        new = html.escape(" ".join(after[j1:j2])) or "—"
        rows += f"<tr><td style='color:#c0392b'>{old}</td><td style='color:#1e8449'>{new}</td></tr>"

display(HTML(f"""
<h3>Where the two transcripts differ</h3>
<table>
  <tr><th>Without prompt (Step 5)</th><th>With prompt</th></tr>
  {rows or "<tr><td colspan='2'>No differences</td></tr>"}
</table>
"""))

Without prompt (Step 5),With prompt
their what,What did
kangaroo,Kangaroo
or something,Something
Maine,Earth
But,—


### Step 5.3: Analyzing the differences

**My analysis** the prompt slightly improved the transcription. A prompt helps most when the audio contains names and specialist terms that Whisper would otherwise guess.

In [55]:
# Step 6: Implementing Audio Chunking
# 6.1 Split audio into manageable chunks

audio_path = "project_nimbus_meeting.mp3"
sample_rate = 16000

# Load the whole meeting as numbers
print(f"🎧 Loading {audio_path}...")
audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
total_duration = len(audio) / sample_rate
print(f"✅ Audio loaded! Total duration: {total_duration:.2f}s ({total_duration / 60:.1f} minutes)")

# Chunk length: 10 minutes, as in the instructions
chunk_duration = 10 * 60  # seconds per chunk

# A short recording would fit in one 10-minute chunk, so use 1-minute chunks to show chunking
if total_duration <= chunk_duration:
    chunk_duration = 60
    print("ℹ️ Recording is shorter than 10 minutes, so 1-minute chunks are used")

# Split audio into chunks
chunk_size = chunk_duration * sample_rate
chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]

print(f"\n🔪 Split into {len(chunks)} chunks of up to {chunk_duration} seconds")

🎧 Loading project_nimbus_meeting.mp3...
✅ Audio loaded! Total duration: 309.20s (5.2 minutes)
ℹ️ Recording is shorter than 10 minutes, so 1-minute chunks are used

🔪 Split into 6 chunks of up to 60 seconds


In [56]:
# 6.2 Process each chunk separately: save each chunk as its own audio file

os.makedirs("chunks", exist_ok=True)

for i, chunk in enumerate(chunks):
    chunk_int16 = np.int16(chunk * 32767)
    chunk_path = f"chunks/chunk_{i+1}.wav"
    wavfile.write(chunk_path, sample_rate, chunk_int16)
    print(f"💾 {chunk_path}: {len(chunk) / sample_rate:.2f}s")

print(f"\n✅ {len(chunks)} chunk files saved in the 'chunks' folder")

💾 chunks/chunk_1.wav: 60.00s
💾 chunks/chunk_2.wav: 60.00s
💾 chunks/chunk_3.wav: 60.00s
💾 chunks/chunk_4.wav: 60.00s
💾 chunks/chunk_5.wav: 60.00s
💾 chunks/chunk_6.wav: 9.20s

✅ 6 chunk files saved in the 'chunks' folder


In [57]:
# 6.3 Combine results with proper timestamps: where each chunk sits in the full recording

chunk_info = []

for i, chunk in enumerate(chunks):
    start = i * chunk_duration                       # chunk offset in the full recording
    end = start + len(chunk) / sample_rate
    chunk_info.append({"chunk": i + 1, "file": f"chunks/chunk_{i+1}.wav", "start": start, "end": end})
    print(f"Chunk {i+1}: [{start:.2f}s - {end:.2f}s]  →  chunks/chunk_{i+1}.wav")

# Checkpoint: the chunks together must cover the whole recording
covered = sum(len(chunk) for chunk in chunks) / sample_rate
print(f"\n⏱️ Total duration: {total_duration:.2f}s | Covered by chunks: {covered:.2f}s")
print("✅ Chunks created correctly" if abs(covered - total_duration) < 0.01 else "⚠ Chunks do not cover the whole recording")

Chunk 1: [0.00s - 60.00s]  →  chunks/chunk_1.wav
Chunk 2: [60.00s - 120.00s]  →  chunks/chunk_2.wav
Chunk 3: [120.00s - 180.00s]  →  chunks/chunk_3.wav
Chunk 4: [180.00s - 240.00s]  →  chunks/chunk_4.wav
Chunk 5: [240.00s - 300.00s]  →  chunks/chunk_5.wav
Chunk 6: [300.00s - 309.20s]  →  chunks/chunk_6.wav

⏱️ Total duration: 309.20s | Covered by chunks: 309.20s
✅ Chunks created correctly


In [58]:
# Step 7: Transcribing Chunks with Timestamps
# 7.1 Transcribe each chunk

chunk_transcripts = []

for i, chunk in enumerate(chunks):
    print(f"🤖 Transcribing chunk {i+1}/{len(chunks)}...")

    # Prepare chunk buffer
    chunk_int16 = np.int16(chunk * 32767)
    buffer = io.BytesIO()
    wavfile.write(buffer, sample_rate, chunk_int16)
    buffer.seek(0)
    buffer.name = f"chunk_{i}.wav"

    # Transcribe with timestamps
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer,
        response_format="verbose_json",
        timestamp_granularities=["segment"]
    )

    chunk_transcripts.append(transcript)
    print(f"Chunk {i+1}: {transcript.text}\n")

🤖 Transcribing chunk 1/6...
Chunk 1: Good morning everyone! This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the Redis cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, Jira number NIM-482, and I expect it to be fixed by Tuesday. Okay, good. Do we need anything from the DevOps team? Yes.

🤖 Transcribing chunk 2/6...
Chunk 2: Yes, we need Tomasz to approve the Terraform changes for the VPC peering. Without that, the pods in the staging namespace cannot reach the legacy database. I can review the Terraform p

In [59]:
# 7.2 Adjust timestamps to account for chunk offset
# 7.3 Combine all transcriptions

all_segments = []

for i, transcript in enumerate(chunk_transcripts):
    offset = chunk_info[i]["start"]  # Whisper starts each chunk at 0s, so add where the chunk begins

    for segment in transcript.segments:
        all_segments.append({
            "start": segment.start + offset,
            "end": segment.end + offset,
            "text": segment.text.strip()
        })

# Combine all transcripts
full_text = " ".join(transcript.text for transcript in chunk_transcripts)

print("📝 Complete transcription with timestamps:")
print("-" * 40)
for segment in all_segments:
    print(f"\n[{segment['start']:.2f}s - {segment['end']:.2f}s]")
    print(f"  {segment['text']}")

📝 Complete transcription with timestamps:
----------------------------------------

[0.00s - 5.80s]
  Good morning everyone! This is the weekly engineering sync for Project Nimbus.

[5.80s - 9.92s]
  Today is Thursday, and we have four items on the agenda.

[9.92s - 18.00s]
  The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update.

[18.00s - 22.40s]
  Priya, do you want to start with the migration?

[22.40s - 28.96s]
  Sure, thanks Marcus. So the Kubernetes migration is about 70% done.

[28.96s - 35.56s]
  We moved the authentication service, and the billing service to the new EKS cluster last week.

[35.56s - 38.56s]
  The main blocker right now is the Redis cache.

[38.56s - 46.36s]
  When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class.

[46.36s - 54.56s]
  I opened a ticket, Jira number NIM-482, and I expect it to be fixed by Tuesday.

[54.56s - 59.96s]
  Okay, good. Do we need anythin

In [60]:
# 7.4 Export with timestamps

os.makedirs("transcriptions", exist_ok=True)

with open("transcriptions/meeting_with_timestamps.txt", "w", encoding="utf-8") as f:
    for segment in all_segments:
        f.write(f"[{segment['start']:.2f}s - {segment['end']:.2f}s] {segment['text']}\n")

print(f"💾 Saved {len(all_segments)} segments to 'transcriptions/meeting_with_timestamps.txt'")

💾 Saved 56 segments to 'transcriptions/meeting_with_timestamps.txt'


In [61]:
# Step 8: Exporting with Timestamps
# 8.1 Create human-readable format with timestamps

def format_readable(seconds):
    """Turn seconds into hh:mm:ss, for example 75.4 -> 00:01:15."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

readable_lines = []
for segment in all_segments:
    line = f"[{format_readable(segment['start'])} - {format_readable(segment['end'])}] {segment['text']}"
    readable_lines.append(line)

print("📝 Human-readable transcript:")
print("-" * 40)
for line in readable_lines:
    print(line)

📝 Human-readable transcript:
----------------------------------------
[00:00:00 - 00:00:05] Good morning everyone! This is the weekly engineering sync for Project Nimbus.
[00:00:05 - 00:00:09] Today is Thursday, and we have four items on the agenda.
[00:00:09 - 00:00:18] The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update.
[00:00:18 - 00:00:22] Priya, do you want to start with the migration?
[00:00:22 - 00:00:28] Sure, thanks Marcus. So the Kubernetes migration is about 70% done.
[00:00:28 - 00:00:35] We moved the authentication service, and the billing service to the new EKS cluster last week.
[00:00:35 - 00:00:38] The main blocker right now is the Redis cache.
[00:00:38 - 00:00:46] When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class.
[00:00:46 - 00:00:54] I opened a ticket, Jira number NIM-482, and I expect it to be fixed by Tuesday.
[00:00:54 - 00:00:59] Okay, good. Do we need anyth

In [62]:
# 8.2 Export to text file

os.makedirs("transcriptions", exist_ok=True)
txt_path = "transcriptions/meeting_transcript.txt"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("MEETING TRANSCRIPT\n")
    f.write(f"Source: {audio_path}\n")
    f.write(f"Duration: {format_readable(total_duration)}\n")
    f.write("=" * 40 + "\n\n")
    for line in readable_lines:
        f.write(line + "\n")

print(f"💾 Saved text transcript to '{txt_path}'")

💾 Saved text transcript to 'transcriptions/meeting_transcript.txt'


In [63]:
# 8.3 Export to SRT (subtitle) format
# Each subtitle: a number, a time line "hh:mm:ss,mmm --> hh:mm:ss,mmm", the text, then an empty line

def format_srt(seconds):
    """Turn seconds into the SRT time format, for example 75.4 -> 00:01:15,400."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    if millis == 1000:  # rounding can reach a full second
        secs, millis = secs + 1, 0
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

srt_path = "transcriptions/meeting_transcript.srt"

with open(srt_path, "w", encoding="utf-8") as f:
    for number, segment in enumerate(all_segments, start=1):
        f.write(f"{number}\n")
        f.write(f"{format_srt(segment['start'])} --> {format_srt(segment['end'])}\n")
        f.write(f"{segment['text']}\n\n")

print(f"💾 Saved {len(all_segments)} subtitles to '{srt_path}'")

💾 Saved 56 subtitles to 'transcriptions/meeting_transcript.srt'


In [64]:
# 8.4 Export to JSON
import json

json_path = "transcriptions/meeting_transcript.json"

transcript_data = {
    "source": audio_path,
    "language": chunk_transcripts[0].language,
    "duration_seconds": round(total_duration, 2),
    "chunk_duration_seconds": chunk_duration,
    "number_of_chunks": len(chunks),
    "segments": [
        {
            "id": number,
            "start": round(segment["start"], 2),
            "end": round(segment["end"], 2),
            "text": segment["text"]
        }
        for number, segment in enumerate(all_segments, start=1)
    ],
    "full_text": full_text
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(transcript_data, f, indent=2, ensure_ascii=False)

print(f"💾 Saved JSON transcript to '{json_path}'")

# Checkpoint: read each file back to verify it is readable
print("\n🔍 Checking the exported files:")
for path in [txt_path, srt_path, json_path]:
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    print(f"\n✅ {path} ({len(content)} characters). First lines:")
    print("\n".join(content.splitlines()[:6]))

# The JSON file must also load back as data
with open(json_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)
print(f"\n✅ JSON loads correctly: {len(loaded['segments'])} segments")

💾 Saved JSON transcript to 'transcriptions/meeting_transcript.json'

🔍 Checking the exported files:

✅ transcriptions/meeting_transcript.txt (5109 characters). First lines:
MEETING TRANSCRIPT
Source: project_nimbus_meeting.mp3
Duration: 00:05:09

[00:00:00 - 00:00:05] Good morning everyone! This is the weekly engineering sync for Project Nimbus.

✅ transcriptions/meeting_transcript.srt (5657 characters). First lines:
1
00:00:00,000 --> 00:00:05,800
Good morning everyone! This is the weekly engineering sync for Project Nimbus.

2
00:00:05,800 --> 00:00:09,920

✅ transcriptions/meeting_transcript.json (12609 characters). First lines:
{
  "source": "project_nimbus_meeting.mp3",
  "language": "english",
  "duration_seconds": 309.2,
  "chunk_duration_seconds": 60,
  "number_of_chunks": 6,

✅ JSON loads correctly: 56 segments
